# Logistic Regression Assignment (Enhanced)

This notebook runs the enhanced supervised learning workflow for personal loan prediction:
- preprocessing improvements
- regularization and class-weight tuning
- threshold tuning
- baseline vs improved comparison
- visualization outputs

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.logistic_regression_loan import (
    DATA_PATH,
    FIGURES_DIR,
    OUTPUT_DIR,
    RANDOM_STATE,
    build_baseline_model,
    build_improved_model,
    build_preprocessor,
    choose_best_threshold,
    evaluate,
    load_data,
    save_coefficient_plot,
    save_confusion_matrix_comparison,
    save_correlation_heatmap,
    save_pr_curve,
    save_roc_curve,
    select_features_for_multicollinearity,
    split_features_target,
    tune_improved_model,
)
from sklearn.base import clone
from sklearn.model_selection import train_test_split

## 1. Load and Inspect Dataset

In [ ]:
df = load_data(DATA_PATH)
print('Shape:', df.shape)
print('Target distribution:')
print(df['Personal Loan'].value_counts())
df.head()

## 2. Split Data and Handle Multicollinearity

In [ ]:
X, y = split_features_target(df)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE
)

selected_cols, high_corr_pairs = select_features_for_multicollinearity(X_train, y_train)
X_train_sel = X_train[selected_cols]
X_val_sel = X_val[selected_cols]
X_test_sel = X_test[selected_cols]

print('Train/Val/Test:', X_train_sel.shape, X_val_sel.shape, X_test_sel.shape)
print('High correlation pairs (|corr|>=0.8):', high_corr_pairs)
print('Selected columns:', selected_cols)

## 3. Train Baseline Model

In [ ]:
preprocessor = build_preprocessor(X_train_sel)
baseline_model = build_baseline_model(preprocessor)
baseline_model.fit(X_train_sel, y_train)

baseline_result, baseline_probs = evaluate(
    baseline_model, X_test_sel, y_test, name='Baseline LR', threshold=0.5
)
pd.Series({
    'accuracy': baseline_result.accuracy,
    'precision': baseline_result.precision,
    'recall': baseline_result.recall,
    'f1': baseline_result.f1,
    'roc_auc': baseline_result.roc_auc,
    'pr_auc': baseline_result.pr_auc,
})

## 4. Train Improved Model (Tuning + Threshold)

In [ ]:
improved_template = build_improved_model(preprocessor)
grid = tune_improved_model(improved_template, X_train_sel, y_train)

improved_model = clone(grid.best_estimator_)
improved_model.fit(X_train_sel, y_train)

best_threshold, threshold_df = choose_best_threshold(improved_model, X_val_sel, y_val)
improved_result, improved_probs = evaluate(
    improved_model, X_test_sel, y_test, name='Improved LR', threshold=best_threshold
)

print('Best params:', grid.best_params_)
print('Best threshold:', best_threshold)

pd.DataFrame([
    {'model': 'Baseline LR', 'threshold': 0.5, 'accuracy': baseline_result.accuracy, 'precision': baseline_result.precision, 'recall': baseline_result.recall, 'f1': baseline_result.f1, 'roc_auc': baseline_result.roc_auc, 'pr_auc': baseline_result.pr_auc},
    {'model': 'Improved LR', 'threshold': best_threshold, 'accuracy': improved_result.accuracy, 'precision': improved_result.precision, 'recall': improved_result.recall, 'f1': improved_result.f1, 'roc_auc': improved_result.roc_auc, 'pr_auc': improved_result.pr_auc},
])

## 5. Generate Visualizations

In [ ]:
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cm_path = save_confusion_matrix_comparison(baseline_result, improved_result)
roc_path = save_roc_curve(y_test, baseline_probs, improved_probs)
pr_path = save_pr_curve(y_test, baseline_probs, improved_probs)
corr_path = save_correlation_heatmap(df)

feature_names = improved_model.named_steps['preprocessor'].get_feature_names_out()
mask = improved_model.named_steps['selector'].get_support()
selected_feature_names = feature_names[mask]
coef_path = save_coefficient_plot(improved_model, selected_feature_names)

print(cm_path)
print(roc_path)
print(pr_path)
print(corr_path)
print(coef_path)

## 6. Display Saved Figures

In [ ]:
for p in [
    cm_path,
    roc_path,
    pr_path,
    corr_path,
    coef_path,
]:
    img = plt.imread(p)
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(Path(p).name)
    plt.show()

## 7. Save Comparison Tables

In [ ]:
comparison_df = pd.DataFrame([
    {
        'model': 'Baseline LR', 'threshold': 0.5,
        'accuracy': baseline_result.accuracy,
        'precision': baseline_result.precision,
        'recall': baseline_result.recall,
        'f1_score': baseline_result.f1,
        'roc_auc': baseline_result.roc_auc,
        'pr_auc': baseline_result.pr_auc,
    },
    {
        'model': 'Improved LR', 'threshold': best_threshold,
        'accuracy': improved_result.accuracy,
        'precision': improved_result.precision,
        'recall': improved_result.recall,
        'f1_score': improved_result.f1,
        'roc_auc': improved_result.roc_auc,
        'pr_auc': improved_result.pr_auc,
    },
])
comparison_df.to_csv(OUTPUT_DIR / 'baseline_vs_improved_metrics.csv', index=False)
threshold_df.to_csv(OUTPUT_DIR / 'threshold_tuning_table.csv', index=False)
comparison_df